# Metric-Guided Content Anonymization — Gemma 3 4B Conservative

## Objective

This notebook investigates an LLM-based approach to content anonymization, with a particular focus on reducing identifying information while preserving the original meaning and relevant content.

Using the Gemma 3 4B Conservative recipe, the experiment follows the same evaluation pipeline as the previous baseline. The conservative prompt asks the model to aim to keep 50% of the transcript unchanged and to retain roughly the same number of utterances. These are generation instructions, not guaranteed constraints; the resulting output must be evaluated.

1. **Conservative generation:** prepare the original transcript and generate an anonymized version using the `gemma-conservative` recipe with Gemma 3 4B.
2. **Document preparation:** load and preprocess both transcripts, examine their structural differences, and prepare them for systematic comparison.
3. **Content segmentation and alignment:** investigate how to divide the original and anonymized documents into comparable text units, using semantic similarity to identify potentially corresponding segments.
4. **Content preservation analysis:** evaluate the differences between the two versions using three complementary metrics:
   - **Term Preservation Index (TPI):** measures the preservation of informative lexical content.
   - **Semantic Similarity:** measures semantic proximity between corresponding text segments.
   - **Natural Language Inference (NLI):** assesses whether the anonymized content entails, is neutral towards, or contradicts the original content.
5. **Information loss analysis:** examine cases where the metrics disagree, identify potentially missing or altered information, and investigate the limitations of each metric.

The experiment uses Brené Brown's TED Talk as a test case, comparing the original transcript with the version generated by the Gemma 3 4B Conservative recipe.

**Overall goal:** establish a reproducible baseline and investigate how complementary evaluation metrics can help identify content degradation and guide future improvements to the anonymization process.

In [1]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import torch

# Locate the project root
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().resolve().parents)
        if (path / "src" / "paths.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root (src/paths.py not found)."
    )

# Add project source directory to Python path
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Experiment directories
DATA_DIR = PROJECT_ROOT / "data" / "aggazzotti_test"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# External repository
EXTERNAL_REPO = (
    PROJECT_ROOT
    / "external"
    / "long-form-speech-anonymization"
)

# Model configuration
MODEL_ID = "google/gemma-3-4b-it"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("External repository:", EXTERNAL_REPO)
print("Model:", MODEL_ID)

Project root: /Users/juni/Documents/interview-anonymization
Data directory: /Users/juni/Documents/interview-anonymization/data/aggazzotti_test
External repository: /Users/juni/Documents/interview-anonymization/external/long-form-speech-anonymization
Model: google/gemma-3-4b-it


In [2]:
# Load the original transcript
TRANSCRIPT_PATH = DATA_DIR / "brene_brown_transcript.txt"

if not TRANSCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"Transcript not found: {TRANSCRIPT_PATH}"
    )

raw_transcript = TRANSCRIPT_PATH.read_text(encoding="utf-8")

print("Source: TED — The Power of Vulnerability")
print("Speaker: Brené Brown")
print("Characters:", len(raw_transcript))
print("Words:", len(raw_transcript.split()))

Source: TED — The Power of Vulnerability
Speaker: Brené Brown
Characters: 17360
Words: 3174


In [3]:
from content.transcript import prepare_transcript

clean_transcript, utterances = prepare_transcript(
    raw_transcript,
    min_words=3,
)

print("Number of utterances:", len(utterances))
print("Total words:", sum(len(u.split()) for u in utterances))

print("\nFirst 10 utterances:")
for i, utterance in enumerate(utterances[:10], start=1):
    print(f"{i}. {utterance}")

Number of utterances: 217
Total words: 3062

First 10 utterances:
1. So, I'll start with this: a couple years ago, an event planner called me because I was going to do a speaking event.
2. And she called, and she said, "I'm really struggling with how to write about you on the little flyer."
3. And I thought, "Well, what's the struggle?"
4. And she said, "Well, I saw you speak, and I'm going to call you a researcher, I think, but I'm afraid if I call you a researcher, no one will come, because they'll think you're boring and irrelevant."
5. And I was like, "Okay."
6. And she said, "But the thing I liked about your talk is you're a storyteller.
7. So I think what I'll do is just call you a storyteller."
8. And of course, the academic, insecure part of me was like, "You're going to call me a what?"
9. And she said, "I'm going to call you a storyteller."
10. And I was like, "Why not 'magic pixie'?"


In [4]:
from content.anonymization_io import (
    build_utterance_dataset,
    save_json,
)

# Dataset metadata
CALL_ID = "synthetic-call-001"
SPEAKER_ID = "speaker-001"
SPEAKER_GENDER = "f"

synthetic_data = build_utterance_dataset(
    utterances=utterances,
    call_id=CALL_ID,
    speaker_id=SPEAKER_ID,
    speaker_gender=SPEAKER_GENDER,
)

DATASET_PATH = DATA_DIR / "brene_brown_utterances.json"

save_json(
    synthetic_data,
    DATASET_PATH,
)

print("Dataset saved:", DATASET_PATH)
print("Number of utterances:", len(utterances))
print("Speaker:", SPEAKER_ID)

Dataset saved: /Users/juni/Documents/interview-anonymization/data/aggazzotti_test/brene_brown_utterances.json
Number of utterances: 217
Speaker: speaker-001


In [5]:
PROMPT_SCRIPT = (
    EXTERNAL_REPO
    / "scripts"
    / "content_anonymization"
    / "generate_paraphrase_prompts.py"
)

if not PROMPT_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Prompt generation script not found: {PROMPT_SCRIPT}"
    )

# Display available arguments
result = subprocess.run(
    [sys.executable, str(PROMPT_SCRIPT), "--help"],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

usage: generate_paraphrase_prompts.py [-h] --utterances UTTERANCES --output
                                      OUTPUT [--model MODEL]
                                      [--recipe {custom,gpt4o-mini,gpt5,gemma,gemma-conservative}]
                                      [--mode {utterance,segment}]
                                      [--segment-by {utterances,tokens}]
                                      [--segment-size-utts SEGMENT_SIZE_UTTS]
                                      [--segment-size-tokens SEGMENT_SIZE_TOKENS]
                                      [--context-prev-utts CONTEXT_PREV_UTTS]
                                      [--endpoint-url ENDPOINT_URL]
                                      [--separator SEPARATOR]
                                      [--system-prompt SYSTEM_PROMPT]
                                      [--include-empty]

Generate paraphrasing batch prompt JSONL.

options:
  -h, --help            show this help message and exit
  --utterances UTTERA

In [6]:
PROMPTS_PATH = DATA_DIR / "brene_brown_gemma_conservative_prompts.jsonl"

if PROMPTS_PATH.exists():
    print(f"Prompts already generated: {PROMPTS_PATH}")
    print("Skipping prompt generation.")

else:
    command = [
        sys.executable,
        str(PROMPT_SCRIPT),
        "--utterances", str(DATASET_PATH),
        "--output", str(PROMPTS_PATH),
        "--model", MODEL_ID,
        "--recipe", "gemma-conservative",
        "--mode", "segment",
        "--segment-by", "utterances",
        "--segment-size-utts", "16",
        "--context-prev-utts", "8",
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        cwd=EXTERNAL_REPO
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Prompt generation failed.")

    print("Prompts generated:", PROMPTS_PATH)

print("File exists:", PROMPTS_PATH.exists())

Prompts already generated: /Users/juni/Documents/interview-anonymization/data/aggazzotti_test/brene_brown_gemma_conservative_prompts.jsonl
Skipping prompt generation.
File exists: True


In [7]:
from content.anonymization_io import load_jsonl

prompt_rows = load_jsonl(PROMPTS_PATH)

print("Number of generated prompts:", len(prompt_rows))

# Inspect the first prompt
first_prompt = prompt_rows[0]

print("\nTop-level keys:")
print(list(first_prompt.keys()))

print("\nFirst prompt:")
print(
    __import__("json").dumps(
        first_prompt,
        indent=2,
        ensure_ascii=False,
    )[:6000]
)

Number of generated prompts: 14

Top-level keys:
['custom_id', 'method', 'url', 'body']

First prompt:
{
  "custom_id": "ted-brene-brown-001-brene-brown-0",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "google/gemma-3-4b-it",
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant that anonymizes utterances. Only provide the paraphrased utterances in the response. Don't include any additional text or explanations in your response."
      },
      {
        "role": "user",
        "content": "Your task is to paraphrase utterances. Follow these instructions exactly.\n\n### CONTEXT ###\n[Beginning of conversation]\n\nSpeaker's gender: f\n\n### UTTERANCES TO PARAPHRASE ###\nSo, I'll start with this: a couple years ago, an event planner called me because I was going to do a speaking event. ## And she called, and she said, \"I'm really struggling with how to write about you on the little flyer.\" ## And I thought, \"Wel

In [8]:
LOCAL_GEMMA_SCRIPT = (
    EXTERNAL_REPO
    / "scripts"
    / "content_anonymization"
    / "run_local_gemma_paraphrase.py"
)

if not LOCAL_GEMMA_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Gemma inference script not found: {LOCAL_GEMMA_SCRIPT}"
    )

# Display available arguments
result = subprocess.run(
    [sys.executable, str(LOCAL_GEMMA_SCRIPT), "--help"],
    capture_output=True,
    text=True,
    cwd=EXTERNAL_REPO,
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Could not inspect the local Gemma script.")

print(result.stdout)

usage: run_local_gemma_paraphrase.py [-h] --prompts PROMPTS --output OUTPUT
                                     [--model-id MODEL_ID]
                                     [--max-new-tokens MAX_NEW_TOKENS]
                                     [--temperature TEMPERATURE]
                                     [--top-p TOP_P]

Run local Gemma paraphrasing from prompt JSONL.

options:
  -h, --help            show this help message and exit
  --prompts PROMPTS     Input prompt JSONL
  --output OUTPUT       Output response JSONL
  --model-id MODEL_ID   HF model ID
  --max-new-tokens MAX_NEW_TOKENS
  --temperature TEMPERATURE
  --top-p TOP_P



In [9]:
# Output file
RESPONSES_PATH = DATA_DIR / "brene_brown_gemma_conservative_responses.jsonl"

# Check inputs
if not PROMPTS_PATH.is_file():
    raise FileNotFoundError(f"Prompts file not found: {PROMPTS_PATH}")

if not LOCAL_GEMMA_SCRIPT.is_file():
    raise FileNotFoundError(f"Gemma script not found: {LOCAL_GEMMA_SCRIPT}")

# Detect available hardware
if torch.cuda.is_available():
    device_info = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    device_info = "Apple Silicon (MPS)"
else:
    device_info = "CPU"

# Run inference only if responses do not already exist
if RESPONSES_PATH.exists():
    print(f"Responses already generated: {RESPONSES_PATH}")
    print("Skipping Gemma inference.")

else:
    command = [
        sys.executable,
        str(LOCAL_GEMMA_SCRIPT),
        "--prompts", str(PROMPTS_PATH),
        "--output", str(RESPONSES_PATH),
        "--model-id", MODEL_ID,
        "--max-new-tokens", "512",
        "--temperature", "0.0",
        "--top-p", "1.0",
    ]

    print("Starting local Gemma 3 4B Conservative inference...")
    print("Model:", MODEL_ID)
    print("Number of prompts:", len(prompt_rows))
    print("Hardware:", device_info)

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        cwd=EXTERNAL_REPO,
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Gemma inference failed.")

    print("Inference completed successfully.")

# Verify output
if not RESPONSES_PATH.is_file():
    raise FileNotFoundError("Gemma inference did not produce an output file.")

print("Responses:", RESPONSES_PATH)
print("File size:", round(RESPONSES_PATH.stat().st_size / 1024, 2), "KB")

Responses already generated: /Users/juni/Documents/interview-anonymization/data/aggazzotti_test/brene_brown_gemma_conservative_responses.jsonl
Skipping Gemma inference.
Responses: /Users/juni/Documents/interview-anonymization/data/aggazzotti_test/brene_brown_gemma_conservative_responses.jsonl
File size: 14.93 KB


In [10]:
from content.anonymization_io import (
    load_jsonl,
    validate_response_rows,
    extract_anonymized_segments,
    reconstruct_utterances,
    reconstruct_document,
    save_text,
)

# Load Gemma 3 4B Conservative responses
response_rows = load_jsonl(RESPONSES_PATH)

print("Number of responses:", len(response_rows))

# Validate inference results
validate_response_rows(response_rows)

# Extract generated segments
anonymized_segments = extract_anonymized_segments(
    response_rows,
    validate=False,
)

# Reconstruct anonymized utterances
anonymized_utterances = reconstruct_utterances(
    anonymized_segments,
    separator="##",
)

# Reconstruct documents
original_document = clean_transcript

anonymized_document = reconstruct_document(
    anonymized_utterances,
    separator="\n",
)

# Save documents
ORIGINAL_DOC_PATH = DATA_DIR / "brene_brown_original_conservative_experiment.txt"
ANONYMIZED_DOC_PATH = DATA_DIR / "brene_brown_anonymized_conservative.txt"

save_text(original_document, ORIGINAL_DOC_PATH)
save_text(anonymized_document, ANONYMIZED_DOC_PATH)

# Summary
print("\nOriginal document:")
print("  Words:", len(original_document.split()))
print("  Utterances:", len(utterances))
print("  File:", ORIGINAL_DOC_PATH)

print("\nAnonymized document:")
print("  Words:", len(anonymized_document.split()))
print("  Generated segments:", len(anonymized_segments))
print("  Reconstructed utterances:", len(anonymized_utterances))
print("  File:", ANONYMIZED_DOC_PATH)

print("\nBoth documents saved successfully.")

Number of responses: 14

Original document:
  Words: 3087
  Utterances: 217
  File: /Users/juni/Documents/interview-anonymization/data/aggazzotti_test/brene_brown_original_conservative_experiment.txt

Anonymized document:
  Words: 2163
  Generated segments: 14
  Reconstructed utterances: 177
  File: /Users/juni/Documents/interview-anonymization/data/aggazzotti_test/brene_brown_anonymized_conservative.txt

Both documents saved successfully.


In [11]:
from IPython.display import HTML, display
from html import escape

display(HTML(f"""
<div style="
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 20px;
">

    <section style="
        border: 1px solid #999;
        border-radius: 8px;
        padding: 15px;
    ">
        <h3>Original — Brené Brown</h3>
        <div style="
            max-height: 600px;
            overflow-y: auto;
            white-space: pre-wrap;
            line-height: 1.6;
            font-size: 13px;
        ">{escape(original_document)}</div>
    </section>

    <section style="
        border: 1px solid #999;
        border-radius: 8px;
        padding: 15px;
    ">
        <h3>Anonymized — Gemma 3 4B Conservative</h3>
        <div style="
            max-height: 600px;
            overflow-y: auto;
            white-space: pre-wrap;
            line-height: 1.6;
            font-size: 13px;
        ">{escape(anonymized_document)}</div>
    </section>

</div>
"""))

In [12]:
from content.diagnostics import (
    compare_documents,
    word_retention,
)

# Compare document statistics
comparison = compare_documents(
    original_document=original_document,
    anonymized_document=anonymized_document,
    original_units=utterances,
    anonymized_units=anonymized_utterances,
)

display(comparison)

# Word retention and reduction
retention_stats = word_retention(
    original_document,
    anonymized_document,
)

print(
    f"\nWord retention: "
    f"{retention_stats['word_retention']:.2%}"
)

print(
    f"Word reduction: "
    f"{retention_stats['word_reduction']:.2%}"
)

,Metric,Original,Anonymized,Change (%)
0,Number of words,3087.000000,2163.000000,-29.93
1,Number of utterances,217.000000,177.000000,-18.43
2,Average words per utterance,14.225806,12.220339,-14.10
3,Document length (characters),16659.000000,12516.000000,-24.87



Word retention: 70.07%
Word reduction: 29.93%


In [13]:
# Load existing MINTS evaluation implementations
from content.evaluation import evaluate_content_pair
from content.embeddings import evaluate_text_pairs
from content.nli import predict_nli

# Centralized model loading
from content.load_models import (
    load_tpi_model,
    load_similarity_model,
    load_nli_model,
)

# --------------------------------------------------
# 1. Model configuration
# --------------------------------------------------

MODEL_NAME = "bert-base-uncased"
SIM_MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"
NLI_MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"

SIMILARITY_THRESHOLD = 0.80
TPI_STRATEGY = "N-TA"
N = 6

DEVICE = torch.device("cpu")

# --------------------------------------------------
# 2. Load evaluation models
# --------------------------------------------------

# Term Preservation Index (TPI)
tokenizer, model, DEVICE = load_tpi_model(
    model_name=MODEL_NAME,
    device=DEVICE,
)

# Semantic similarity
sim_model = load_similarity_model(
    model_name=SIM_MODEL_NAME,
)

# Natural Language Inference (NLI)
nli_tokenizer, nli_model, NLI_DEVICE, NLI_LABEL_MAPPING = (
    load_nli_model(
        model_name=NLI_MODEL_NAME,
        device="cpu",
    )
)

# --------------------------------------------------
# Summary
# --------------------------------------------------

print("All evaluation models loaded successfully.")
print(f"TPI model: {MODEL_NAME} ({DEVICE})")
print(f"Semantic similarity model: {SIM_MODEL_NAME}")
print(f"NLI model: {NLI_MODEL_NAME} ({NLI_DEVICE})")

/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1104 18:48:54.377000 22982 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 6700.54it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights

All evaluation models loaded successfully.
TPI model: bert-base-uncased (cpu)
Semantic similarity model: sentence-transformers/paraphrase-mpnet-base-v2
NLI model: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli (cpu)


In [14]:
from content.diagnostics import token_length_audit

texts = {
    "Original": original_document,
    "Anonymized": anonymized_document,
}

tokenizer_comparison = token_length_audit(
    texts=texts,
    tokenizers={
        "BERT": tokenizer,
        "NLI": nli_tokenizer,
    },
    add_special_tokens=True,
)

display(tokenizer_comparison)

print(
    f"\nMPNet maximum sequence length: "
    f"{sim_model.max_seq_length}"
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4125 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4047 > 512). Running this sequence through the model will result in indexing errors


,Text,BERT tokens,NLI tokens
0,Original,4125,4047
1,Anonymized,2582,2524



MPNet maximum sequence length: 512


In [15]:
from content.transcript import prepare_sentence_units

original_sentences = prepare_sentence_units(
    original_document,
    language="english",
)

anonymized_units = anonymized_utterances

print("ORIGINAL")
print("Number of sentences:", len(original_sentences))

print("\nFirst 5 sentences:")
for i, sentence in enumerate(original_sentences[:5], start=1):
    print(f"{i}. {sentence}")

print("\nANONYMIZED")
print("Number of generated units:", len(anonymized_units))

print("\nFirst 5 units:")
for i, unit in enumerate(anonymized_units[:5], start=1):
    print(f"{i}. {unit}")

ORIGINAL
Number of sentences: 221

First 5 sentences:
1. So, I'll start with this: a couple years ago, an event planner called me because I was going to do a speaking event.
2. And she called, and she said, "I'm really struggling with how to write about you on the little flyer."
3. And I thought, "Well, what's the struggle?"
4. And she said, "Well, I saw you speak, and I'm going to call you a researcher, I think, but I'm afraid if I call you a researcher, no one will come, because they'll think you're boring and irrelevant."
5. And I was like, "Okay."

ANONYMIZED
Number of generated units: 177

First 5 units:
1. i started by describing an experience a few years back when a conference organizer contacted me about a speaking engagement
2. she needed help crafting my bio for a flyer
3. she worried that labeling me as a researcher would scare people away, thinking it sounded dull
4. but she pointed out i’m a natural storyteller
5. so she decided to call me a storyteller


### Selected segmentation method: Optimized proportional chunking

The initial proportional segmentation provides a chronological correspondence between the original and anonymized documents.

The number of chunks is estimated dynamically from the total number of BERT tokens in the original document, using a target of 150 tokens per chunk.

To improve the initial alignment, we apply a local boundary optimization strategy:

1. **Initial segmentation:** divide both documents proportionally into the estimated number of chunks.
2. **Local boundary search:** keep the original document boundaries fixed and allow anonymized document boundaries to move within a window of ±2 units around their initial proportional positions.
3. **Similarity optimization:** select the candidate boundary that maximizes the sum of cosine similarities between the two adjacent corresponding chunk pairs.

Sentence embeddings are computed using `paraphrase-mpnet-base-v2`.

The optimized segmentation is compared against the proportional baseline using the same number of chunks and the same semantic similarity metric.

The optimized segmentation is retained for subsequent content preservation analyses.

In [16]:
import math

from content.proportional_chunking import (
    build_proportional_chunks,
    optimize_proportional_boundaries,
    estimate_n_chunks,
)

TARGET_TOKENS_PER_CHUNK = 150
BOUNDARY_WINDOW = 2
MIN_CHUNK_SIZE = 2

# Estimate the number of chunks from original token count
chunk_config = estimate_n_chunks(
    original_units=original_sentences,
    tokenizer=tokenizer,
    target_tokens_per_chunk=TARGET_TOKENS_PER_CHUNK,
)

N_CHUNKS = chunk_config["n_chunks"]

# Convert target number of chunks into units per chunk
ORIGINAL_CHUNK_SIZE = math.ceil(
    len(original_sentences) / N_CHUNKS
)

print("Chunk configuration:", chunk_config)
print("Original units per baseline chunk:", ORIGINAL_CHUNK_SIZE)

# Proportional baseline
proportional_chunks_df = build_proportional_chunks(
    original_units=original_sentences,
    anonymized_units=anonymized_units,
    similarity_model=sim_model,
    original_chunk_size=ORIGINAL_CHUNK_SIZE,
)

# Optimized proportional chunking
optimized_chunks_df = optimize_proportional_boundaries(
    original_units=original_sentences,
    anonymized_units=anonymized_units,
    similarity_model=sim_model,
    n_chunks=N_CHUNKS,
    window=BOUNDARY_WINDOW,
    min_chunk_size=MIN_CHUNK_SIZE,
)

# Validate chunk counts
assert len(proportional_chunks_df) == N_CHUNKS
assert len(optimized_chunks_df) == N_CHUNKS

# Validate complete coverage
assert proportional_chunks_df["original_size"].sum() == len(original_sentences)
assert proportional_chunks_df["anonymized_size"].sum() == len(anonymized_units)

assert optimized_chunks_df["original_size"].sum() == len(original_sentences)
assert optimized_chunks_df["anonymized_size"].sum() == len(anonymized_units)

print("Proportional chunks:", len(proportional_chunks_df))
print("Optimized chunks:", len(optimized_chunks_df))
print("All original and anonymized units are included.")

Chunk configuration: {'total_tokens': 4123, 'target_tokens_per_chunk': 150, 'n_chunks': 28, 'expected_average_tokens': 147.25}
Original units per baseline chunk: 8
Proportional chunks: 28
Optimized chunks: 28
All original and anonymized units are included.


In [17]:
comparison_df = pd.DataFrame({
    "Proportional": proportional_chunks_df["similarity"].describe(),
    "Optimized": optimized_chunks_df["similarity"].describe(),
})

display(comparison_df.round(4))

baseline_mean = proportional_chunks_df["similarity"].mean()
optimized_mean = optimized_chunks_df["similarity"].mean()

print(f"Baseline mean similarity:  {baseline_mean:.4f}")
print(f"Optimized mean similarity: {optimized_mean:.4f}")
print(f"Absolute improvement:      {optimized_mean - baseline_mean:+.4f}")
print(
    f"Relative improvement:      "
    f"{(optimized_mean / baseline_mean - 1) * 100:+.2f}%"
)

for _, chunk in optimized_chunks_df.iterrows():

    print(f"\n{'=' * 90}")
    print(
        f"CHUNK {chunk['chunk_id']} | "
        f"Similarity: {chunk['similarity']:.4f}"
    )

    print(
        f"Original: {chunk['original_size']} units | "
        f"Anonymized: {chunk['anonymized_size']} units"
    )

    print("\nORIGINAL:")
    print(chunk["original_text"])

    print("\nANONYMIZED:")
    print(chunk["anonymized_text"])

,Proportional,Optimized
count,28.0000,28.0000
mean,0.7499,0.8382
std,0.0877,0.0751
min,0.5402,0.6480
25%,0.6872,0.7900
50%,0.7629,0.8464
75%,0.8058,0.8805
max,0.8778,0.9931


Baseline mean similarity:  0.7499
Optimized mean similarity: 0.8382
Absolute improvement:      +0.0883
Relative improvement:      +11.77%

CHUNK 0 | Similarity: 0.7344
Original: 7 units | Anonymized: 5 units

ORIGINAL:
So, I'll start with this: a couple years ago, an event planner called me because I was going to do a speaking event. And she called, and she said, "I'm really struggling with how to write about you on the little flyer." And I thought, "Well, what's the struggle?" And she said, "Well, I saw you speak, and I'm going to call you a researcher, I think, but I'm afraid if I call you a researcher, no one will come, because they'll think you're boring and irrelevant." And I was like, "Okay." And she said, "But the thing I liked about your talk is you're a storyteller. So I think what I'll do is just call you a storyteller."

ANONYMIZED:
i started by describing an experience a few years back when a conference organizer contacted me about a speaking engagement she needed help craf

In [18]:
# Check token lengths for all optimized chunks

length_records = []

for _, chunk in optimized_chunks_df.iterrows():

    original_text = chunk["original_text"]
    anonymized_text = chunk["anonymized_text"]

    original_bert_tokens = len(
        tokenizer(original_text, add_special_tokens=True)["input_ids"]
    )

    anonymized_bert_tokens = len(
        tokenizer(anonymized_text, add_special_tokens=True)["input_ids"]
    )

    nli_pair_tokens = len(
        nli_tokenizer(
            anonymized_text,
            original_text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )

    length_records.append({
        "chunk_id": chunk["chunk_id"],
        "original_bert_tokens": original_bert_tokens,
        "anonymized_bert_tokens": anonymized_bert_tokens,
        "nli_pair_tokens": nli_pair_tokens,
        "similarity": chunk["similarity"],
    })

chunk_lengths_df = pd.DataFrame(length_records)

display(chunk_lengths_df)

print("Maximum original BERT tokens:",
      chunk_lengths_df["original_bert_tokens"].max())

print("Maximum anonymized BERT tokens:",
      chunk_lengths_df["anonymized_bert_tokens"].max())

print("Maximum NLI pair tokens:",
      chunk_lengths_df["nli_pair_tokens"].max())

print("Mean similarity:",
      chunk_lengths_df["similarity"].mean())

,chunk_id,original_bert_tokens,anonymized_bert_tokens,nli_pair_tokens,similarity
0,0,176,71,237,0.734356
1,1,141,73,200,0.835794
2,2,187,150,333,0.910010
3,3,166,171,334,0.993056
4,4,137,83,219,0.856963
5,5,182,109,284,0.901880
6,6,189,109,289,0.915934
7,7,207,108,300,0.881937
8,8,238,179,411,0.933782
9,9,112,72,183,0.914794


Maximum original BERT tokens: 259
Maximum anonymized BERT tokens: 179
Maximum NLI pair tokens: 411
Mean similarity: 0.8381824770144054


In [19]:
from content.evaluation import evaluate_content_pair
from content.embeddings import evaluate_text_pairs
from content.nli import predict_nli

# --------------------------------------------------
# 1. Prepare chunk pairs
# --------------------------------------------------

chunk_pairs = optimized_chunks_df[
    ["chunk_id", "original_text", "anonymized_text"]
].reset_index(drop=True)

original_texts = chunk_pairs["original_text"].tolist()
anonymized_texts = chunk_pairs["anonymized_text"].tolist()

print(f"Number of chunk pairs: {len(chunk_pairs)}")


# --------------------------------------------------
# 2. TPI evaluation
# --------------------------------------------------

tpi_records = []

for _, chunk in chunk_pairs.iterrows():

    result = evaluate_content_pair(
        original_text=chunk["original_text"],
        anonymized_text=chunk["anonymized_text"],
        tokenizer=tokenizer,
        model=model,
        device=DEVICE,
        similarity_threshold=SIMILARITY_THRESHOLD,
        tpi_strategy=TPI_STRATEGY,
        N=N,
    )

    tpi_records.append({
        "chunk_id": chunk["chunk_id"],
        "TPI": result["TPI"],
        "L_TPI": result["L_TPI"],
        "TIC_original": result["tpi"]["TIC_original"],
        "information_lost": result["tpi"]["information_lost"],
        "n_affected_terms": len(
            result["alignment"]["affected_original_indices"]
        ),
    })

tpi_df = pd.DataFrame(tpi_records)


# --------------------------------------------------
# 3. Semantic similarity evaluation
# --------------------------------------------------

similarity_df = evaluate_text_pairs(
    model=sim_model,
    pairs_by_category={
        "optimized": list(zip(original_texts, anonymized_texts))
    },
)

similarity_df = similarity_df[["cosine"]].rename(
    columns={"cosine": "semantic_similarity"}
)

similarity_df.insert(
    0,
    "chunk_id",
    chunk_pairs["chunk_id"],
)


# --------------------------------------------------
# 4. NLI evaluation — both directions
# --------------------------------------------------

# Original as premise, anonymized as hypothesis:
# Are the anonymized claims supported by the original?

original_to_anonymized_pairs = list(
    zip(original_texts, anonymized_texts)
)

# Anonymized as premise, original as hypothesis:
# Is the original information preserved in the anonymized text?

anonymized_to_original_pairs = list(
    zip(anonymized_texts, original_texts)
)

# Check that no pair will be truncated by the NLI model
max_nli_length = nli_tokenizer.model_max_length

if chunk_lengths_df["nli_pair_tokens"].max() > max_nli_length:
    raise ValueError(
        "At least one NLI pair exceeds the model's maximum "
        f"sequence length ({max_nli_length} tokens)."
    )

nli_original_to_anonymized = predict_nli(
    tokenizer=nli_tokenizer,
    model=nli_model,
    pairs=original_to_anonymized_pairs,
    device=NLI_DEVICE,
    label_mapping=NLI_LABEL_MAPPING,
    batch_size=8,
)

nli_anonymized_to_original = predict_nli(
    tokenizer=nli_tokenizer,
    model=nli_model,
    pairs=anonymized_to_original_pairs,
    device=NLI_DEVICE,
    label_mapping=NLI_LABEL_MAPPING,
    batch_size=8,
)

# Add direction-specific prefixes
nli_original_to_anonymized = (
    nli_original_to_anonymized
    .add_prefix("original_to_anonymized_")
)

nli_anonymized_to_original = (
    nli_anonymized_to_original
    .add_prefix("anonymized_to_original_")
)

nli_df = pd.concat(
    [
        nli_original_to_anonymized,
        nli_anonymized_to_original,
    ],
    axis=1,
)

nli_df.insert(
    0,
    "chunk_id",
    chunk_pairs["chunk_id"],
)


# --------------------------------------------------
# 5. Merge all metrics
# --------------------------------------------------

metrics_df = (
    tpi_df
    .merge(similarity_df, on="chunk_id")
    .merge(nli_df, on="chunk_id")
)

# Most likely NLI class in each direction
metrics_df["original_to_anonymized_prediction"] = (
    metrics_df[
        [
            "original_to_anonymized_entailment",
            "original_to_anonymized_neutral",
            "original_to_anonymized_contradiction",
        ]
    ]
    .idxmax(axis=1)
    .str.replace("original_to_anonymized_", "", regex=False)
)

metrics_df["anonymized_to_original_prediction"] = (
    metrics_df[
        [
            "anonymized_to_original_entailment",
            "anonymized_to_original_neutral",
            "anonymized_to_original_contradiction",
        ]
    ]
    .idxmax(axis=1)
    .str.replace("anonymized_to_original_", "", regex=False)
)

print("Metric evaluation completed.")
print("Number of evaluated chunks:", len(metrics_df))

display(metrics_df.round(4))

Number of chunk pairs: 28
Metric evaluation completed.
Number of evaluated chunks: 28


,chunk_id,TPI,L_TPI,TIC_original,information_lost,n_affected_terms,semantic_similarity,original_to_anonymized_entailment,original_to_anonymized_neutral,original_to_anonymized_contradiction,anonymized_to_original_entailment,anonymized_to_original_neutral,anonymized_to_original_contradiction,original_to_anonymized_prediction,anonymized_to_original_prediction
0,0,0.1852,0.8148,265.5501,216.3834,108,0.7344,0.9985,0.0012,0.0001,0.9790,0.0202,0.0007,entailment,entailment
1,1,0.1782,0.8218,250.8692,206.1561,81,0.8358,0.9985,0.0011,0.0006,0.3635,0.5913,0.0451,entailment,neutral
2,2,0.7261,0.2739,279.2968,76.4857,24,0.9100,0.9819,0.0111,0.0070,0.4961,0.2959,0.2080,entailment,entailment
3,3,0.8119,0.1881,300.9033,56.5965,7,0.9931,0.0611,0.9365,0.0023,0.9897,0.0075,0.0025,neutral,entailment
4,4,0.1842,0.8158,211.0940,172.2149,86,0.8570,0.9980,0.0019,0.0001,0.0101,0.9893,0.0007,entailment,neutral
5,5,0.2189,0.7811,331.5873,259.0086,100,0.9019,0.9990,0.0009,0.0001,0.6489,0.3464,0.0045,entailment,entailment
6,6,0.2920,0.7080,351.8839,249.1224,111,0.9159,0.9980,0.0020,0.0001,0.7637,0.2341,0.0020,entailment,entailment
7,7,0.1005,0.8995,395.2455,355.5222,117,0.8819,0.9995,0.0006,0.0001,0.8604,0.1367,0.0027,entailment,entailment
8,8,0.3630,0.6370,391.2772,249.2285,125,0.9338,0.9961,0.0035,0.0002,0.9932,0.0068,0.0002,entailment,entailment
9,9,0.0821,0.9179,156.7850,143.9073,61,0.9148,0.9990,0.0008,0.0001,0.9976,0.0023,0.0001,entailment,entailment


In [20]:
# ==================================================
# 1. DESCRIPTIVE STATISTICS
# ==================================================

main_metrics = [
    "TPI",
    "L_TPI",
    "semantic_similarity",
]

nli_probability_columns = [
    "original_to_anonymized_entailment",
    "original_to_anonymized_neutral",
    "original_to_anonymized_contradiction",
    "anonymized_to_original_entailment",
    "anonymized_to_original_neutral",
    "anonymized_to_original_contradiction",
]

diagnostic_columns = [
    "TIC_original",
    "information_lost",
    "n_affected_terms",
]

descriptive_columns = (
    main_metrics
    + nli_probability_columns
    + diagnostic_columns
)

descriptive_stats = (
    metrics_df[descriptive_columns]
    .describe()
    .T
    .round(4)
)

display(descriptive_stats)


# ==================================================
# 2. NLI PREDICTION DISTRIBUTIONS
# ==================================================

print("NLI predictions: Original → Anonymized")

original_to_anonymized_counts = (
    metrics_df["original_to_anonymized_prediction"]
    .value_counts()
    .reindex(["entailment", "neutral", "contradiction"], fill_value=0)
)

original_to_anonymized_distribution = pd.DataFrame({
    "count": original_to_anonymized_counts,
    "percentage": (
        original_to_anonymized_counts / len(metrics_df) * 100
    ),
})

display(original_to_anonymized_distribution.round(2))


print("NLI predictions: Anonymized → Original")

anonymized_to_original_counts = (
    metrics_df["anonymized_to_original_prediction"]
    .value_counts()
    .reindex(["entailment", "neutral", "contradiction"], fill_value=0)
)

anonymized_to_original_distribution = pd.DataFrame({
    "count": anonymized_to_original_counts,
    "percentage": (
        anonymized_to_original_counts / len(metrics_df) * 100
    ),
})

display(anonymized_to_original_distribution.round(2))


# ==================================================
# 3. NLI AVERAGE PROBABILITIES BY DIRECTION
# ==================================================

nli_probability_summary = pd.DataFrame(
    {
        "Original → Anonymized": [
            metrics_df["original_to_anonymized_entailment"].mean(),
            metrics_df["original_to_anonymized_neutral"].mean(),
            metrics_df["original_to_anonymized_contradiction"].mean(),
        ],
        "Anonymized → Original": [
            metrics_df["anonymized_to_original_entailment"].mean(),
            metrics_df["anonymized_to_original_neutral"].mean(),
            metrics_df["anonymized_to_original_contradiction"].mean(),
        ],
    },
    index=["Entailment", "Neutral", "Contradiction"],
)

display(nli_probability_summary.round(4))


# ==================================================
# 4. EXPLORATORY CORRELATIONS
# ==================================================

correlation_columns = [
    "TPI",
    "semantic_similarity",
    "original_to_anonymized_entailment",
    "original_to_anonymized_neutral",
    "anonymized_to_original_entailment",
    "anonymized_to_original_neutral",
]

correlation_matrix = metrics_df[
    correlation_columns
].corr(method="spearman")

display(correlation_matrix.round(3))

,count,mean,std,min,25%,50%,75%,max
TPI,28.0,0.1885,0.1865,0.0262,0.0772,0.1436,0.2066,0.8119
L_TPI,28.0,0.8115,0.1865,0.1881,0.7934,0.8564,0.9228,0.9738
semantic_similarity,28.0,0.8382,0.0751,0.6480,0.7900,0.8464,0.8805,0.9931
original_to_anonymized_entailment,28.0,0.9272,0.2519,0.0111,0.9956,0.9983,0.9990,0.9995
original_to_anonymized_neutral,28.0,0.0718,0.2517,0.0005,0.0008,0.0014,0.0040,0.9883
original_to_anonymized_contradiction,28.0,0.0010,0.0030,0.0001,0.0001,0.0001,0.0003,0.0149
anonymized_to_original_entailment,28.0,0.5998,0.3691,0.0049,0.3114,0.6562,0.9738,0.9985
anonymized_to_original_neutral,28.0,0.3479,0.3638,0.0015,0.0261,0.1854,0.6309,0.9907
anonymized_to_original_contradiction,28.0,0.0524,0.1570,0.0001,0.0007,0.0023,0.0054,0.7197
TIC_original,28.0,260.8215,105.3497,41.6828,187.3294,267.8781,323.3506,473.2456


NLI predictions: Original → Anonymized


,count,percentage
original_to_anonymized_prediction,,
entailment,26,92.86
neutral,2,7.14
contradiction,0,0.00


NLI predictions: Anonymized → Original


,count,percentage
anonymized_to_original_prediction,,
entailment,18,64.29
neutral,9,32.14
contradiction,1,3.57


,Original → Anonymized,Anonymized → Original
Entailment,0.9272,0.5998
Neutral,0.0718,0.3479
Contradiction,0.0010,0.0524


,TPI,semantic_similarity,original_to_anonymized_entailment,original_to_anonymized_neutral,anonymized_to_original_entailment,anonymized_to_original_neutral
TPI,1.000,0.352,-0.106,0.141,-0.030,0.047
semantic_similarity,0.352,1.000,0.130,-0.138,0.227,-0.164
original_to_anonymized_entailment,-0.106,0.130,1.000,-0.987,0.153,-0.013
original_to_anonymized_neutral,0.141,-0.138,-0.987,1.000,-0.183,0.049
anonymized_to_original_entailment,-0.030,0.227,0.153,-0.183,1.000,-0.937
anonymized_to_original_neutral,0.047,-0.164,-0.013,0.049,-0.937,1.000
